# 3-1회차 실습 | 펭귄, 이번엔 모델 말고 데이터

1-2때 손으로 규칙 만들어서 94.44% 찍었음.
2-1때 제대로 나눠서 재보니 KNN만 혼자 `0.7801`로 처박혔음. 이유는 들었는데 못 고쳤음.

> **이 문제는 오늘 못 고침. 스케일링(전처리)을 배워야 고칠 수 있음.**

오늘 그거 고침. 그리고 하나 더 봄 — **고친 결과를 믿어도 되는지.**

### 오늘 모델은 안 배움

`model` 하나를 만들어서 줌. 그건 그대로 쓰면 됨.

> 숫자의 크기 차이에 **영향을 받는 모델**이라는 것만 알면 됨. 원리는 뒤에서 배움.

**나머지는 직접 짬.** 오늘 배운 `Pipeline`, `ColumnTransformer` 를 손으로 조립함.

---
## 실습 규칙

1. `[예측]` 나오면 **실행 전에** 적을 것
2. `[키보드 금지]` 나오면 코드 치지 말고 생각만 할 것
3. **예측 회수** 칸을 비우고 넘어가지 말 것
4. 점수가 좋아졌다고 끝이 아님. **절차도 볼 것**

## 오늘 쓰는 것

| 이름 | 하는 일 |
|---|---|
| `train_df`, `test_df` | 342마리를 8:2로 나눈 것 (2-1과 동일, `random_state=7`) |
| `MEAS` | 측정값 컬럼 4개 |
| `model` | 오늘 쓸 모델. **그대로 씀** |
| `diag_train_df`, `diag_val_df` | Q5용. `train` 을 다시 나눈 것 |

> `penguins.csv`가 같은 폴더에 있어야 함.
> **Q5 예측을 적기 전에 아래로 스크롤하지 말 것** — 답이 보임.

In [ ]:
# 준비 과정입니다. 읽지 말고 그대로 실행만 하세요.
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

MEAS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# google drive에 있음. 귀찮으면 그냥 인터넷으로 데이터 로드해도 되니, csv 다운로드 안받아도 됨.
# 같은 폴더의 penguins.csv 를 씁니다. 없으면 seaborn 에서 받아옵니다(네트워크 필요).
if os.path.exists("penguins.csv"):
    peng = pd.read_csv("penguins.csv")
else:
    import seaborn as sns
    print("[안내] penguins.csv 가 없어서 seaborn 에서 내려받습니다. 네트워크가 필요합니다.")
    peng = sns.load_dataset("penguins")

peng = peng.dropna(subset=MEAS).reset_index(drop=True)
y = peng["species"].values

# 2-1 과 완전히 동일한 분할
i_tr, i_te = train_test_split(np.arange(len(peng)), test_size=0.2, stratify=y, random_state=7)
train_df, test_df = peng.iloc[i_tr], peng.iloc[i_te]
y_train, y_test = y[i_tr], y[i_te]

# Q5 전용 - final test 69 는 열지 않고, train 안에서 다시 나눕니다.
j_a, j_b = train_test_split(np.arange(len(train_df)), test_size=0.2,
                            stratify=y_train, random_state=7)
diag_train_df, diag_val_df = train_df.iloc[j_a], train_df.iloc[j_b]
y_diag_train, y_diag_val = y_train[j_a], y_train[j_b]

# 오늘 쓸 모델. 원리는 아직 안 배웠으니 이건 그대로 씁니다.
# 숫자의 크기 차이에 영향을 받는 모델이라는 것만 알면 됩니다.
model = KNeighborsClassifier(5)

assert (len(peng), len(train_df), len(test_df)) == (342, 273, 69)
assert (len(diag_train_df), len(diag_val_df)) == (218, 55)

print(f"펭귄 {len(peng)}마리")
print(f"  -> train {len(train_df)} / final test {len(test_df)} (final test 는 오늘 열지 않음)")
print(f"  -> Q5용: train {len(train_df)} = diagnostic_train {len(diag_train_df)} + diagnostic_val {len(diag_val_df)}")
print(f"결측: {peng.isna().sum()[lambda s: s > 0].to_dict()}  (오늘 sex 는 쓰지 않습니다)")

---
## Q1. 이상한 숫자를 찾아라  `[키보드 금지]`

아래 셀을 실행해서 측정값 4개의 범위를 볼 것. **그다음 코드는 치지 말고 생각만 할 것.**

In [ ]:
train_df[MEAS].describe().T[["min", "max"]].assign(범위=lambda d: d["max"] - d["min"]).round(1)

### 답할 것 (코드 없이)

**Q1-1.** 네 컬럼의 범위를 적을 것.

| 컬럼 | min | max | 범위 |
|---|---|---|---|
| bill_length_mm | | | |
| bill_depth_mm | | | |
| flipper_length_mm | | | |
| body_mass_g | | | |

**Q1-2.** `body_mass_g`를 **g 대신 kg**로 바꾼다고 해보자. (1000으로 나누면 됨)

- `body_mass`의 범위는 얼마가 되나?
- 그러면 `bill_depth_mm`의 범위(8.4)와 비교했을 때 **어느 쪽이 더 커지나?**

- 답:

**Q1-3.** Q1-2에서 뭘 알 수 있나?

> 펭귄은 한 마리도 안 바뀌었음. 몸무게를 **1000으로 나눠서 적었을 뿐**임.
> 그런데 **어느 컬럼이 "큰 숫자"인지는 뒤바뀌었음.**

- 답:

**Q1-4.** 이 데이터를 **그대로** 모델에 넣으면 어떨 것 같은가?

> 확신하지 않아도 됨. **"문제가 될 것 같다 / 아닐 것 같다"와 그 이유**만 적을 것.
> 맞는지는 Q2에서 확인함.

- 답:

---
## Q2. 같은 모델, 다른 데이터

2-1에서 이랬음.

| 모델 | CV 평균 | CV 표준편차 |
|---|---|---|
| LogisticRegression | 0.9927 | 0.0089 |
| DecisionTree | 0.9671 | 0.0241 |
| **KNN (k=5)** | **0.7801** | 0.0355 |

**[예측]** Q1에서 본 범위 차이를 없애면 `0.7801`이 얼마까지 갈 것 같음? ______

In [ ]:
# TODO: 두 가지를 비교할 것. 모델은 위에서 만든 model 을 그대로 씀.
#
#   A. 전처리 없이            model 만 교차검증
#   B. StandardScaler + model 을 Pipeline 으로 묶어서 교차검증
#
#   둘 다 cross_val_score(..., cv=5) 로 train_df[MEAS], y_train 을 씀.
#   평균과 표준편차를 출력할 것.
#
# Pipeline 짜는 법은 본강의에서 봤음. 단계마다 이름을 붙여서 리스트로 넘김.

### 답할 것

| 방식 | CV 평균 | CV 표준편차 |
|---|---|---|
| A. 그대로 | | |
| B. 스케일 맞춤 | | |
| 차이 | | |

**예측 회수**

| | 값 |
|---|---|
| 내 예측 | |
| 실제 | |
| 빗나간 폭 | |

**Q2-1.** **모델은 안 바꿨음.** 그럼 무엇만 바꿨나?

- 답:

**Q2-2.** `0.7801`이 2-1과 소수점까지 같음. 왜 같을까?

- 답:

**Q2-3.** Q1-4에서 적은 예상이 맞았나? 틀렸다면 무엇을 잘못 봤나?

- 답:

---
## Q3. 점수가 같으면 아무거나 써도 되나?

이번엔 `island`를 넣어봄. 글자라서 숫자로 바꿔야 함. 두 가지 방법을 비교함.

- **A**: 섬마다 숫자 하나씩 붙이기 (이미 짜여 있음, 실행만 할 것)
- **B**: One-Hot

**[예측]** 두 방법의 점수는? ______ (A가 높다 / B가 높다 / 같다)

In [ ]:
# 방법 A - 이미 짜여 있는 코드입니다. 실행만 하세요.
prep_A = ColumnTransformer([
    ("num", StandardScaler(), MEAS),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["island"]),
])
pipe_A = Pipeline([("prep", prep_A), ("model", model)])
scores_A = cross_val_score(pipe_A, train_df[MEAS + ["island"]], y_train, cv=5)

print(f"A : {scores_A.mean():.4f} +- {scores_A.std():.4f}")
print("각 섬에 붙은 숫자:", dict(zip(sorted(train_df["island"].unique()), range(3))))

In [ ]:
# TODO: 방법 B - island 를 One-Hot 으로 표현했을 때의 점수를 구할 것.
#
#   위 A 코드를 그대로 복사해서 **딱 한 군데만** 바꾸면 됨.
#   OrdinalEncoder(...) 자리에 OneHotEncoder(handle_unknown="ignore") 를 넣을 것.
#   (handle_unknown 은 "처음 보는 섬이 나오면 어떻게 할래?" 를 정하는 옵션임. 그대로 쓰면 됨)
#
#   변수 이름은 prep_B, pipe_B, scores_B 로 할 것.

### 답할 것

| 방법 | CV 평균 | CV 표준편차 |
|---|---|---|
| A. 섬마다 숫자 하나 | | |
| B. One-Hot | | |

**예측 회수** — 예측이 맞았나?

- 답:

**Q3-1.** A에서 각 섬에 붙은 숫자를 적을 것.

- Biscoe = , Dream = , Torgersen =

**Q3-2.** 두 점수를 비교하면?

- 답:

**Q3-3.** **점수가 같으니까 두 방법 다 괜찮다고 결론 내려도 되나?**

> Q3-1에 적은 숫자들을 다시 볼 것. 그 숫자들 사이에 **어떤 관계**가 생겼나?
> 그 관계가 **실제 섬에 존재하나?**

- 답:

**Q3-4.** 1-2 Q13에서도 `island`가 나왔음. 그때 지적한 문제와 Q3-3의 문제는 **같은 문제인가 다른 문제인가?**

- 답:

---
## Q3 확장. 조합을 한꺼번에 확인해보기

지금까지 하나씩 비교했음.

- Q2: 스케일링 없음 vs StandardScaler
- Q3: island를 숫자 하나 vs One-Hot

근데 스케일러는 세 개였고(Standard / MinMax / Robust), 인코더는 두 개임.
**조합이 6개**임. 하나씩 손으로 돌리면 여섯 번 짜야 함.

2-1에서 이럴 때 쓰는 도구를 배웠음. **GridSearchCV**.

> 2-1에서 이렇게 말했음.
> **"GridSearchCV는 '설계자'가 아니라 '탐색기'임. 내가 넣은 후보 안에서만 비교함."**

오늘은 모델 설정이 아니라 **전처리 후보**를 넣어볼 것. 모델은 `KNeighborsClassifier(5)`로 고정임.

**[예측]** 6개 조합 중 1위는 몇 개일 것 같음? ______ 개

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, RobustScaler

scalers  = [StandardScaler(), MinMaxScaler(), RobustScaler()]
encoders = [OneHotEncoder(handle_unknown="ignore"),
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)]


# 뼈대는 Q3 에서 짠 것과 똑같음. 이미 만들어놨음.
# 단계마다 붙은 이름표를 잘 볼 것. "num", "cat", "prep", "model".
prep = ColumnTransformer([
    ("num", StandardScaler(), MEAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["island"]),
])
pipe = Pipeline([("prep", prep), ("model", model)])


# TODO: param_grid 를 채울 것.
#   위에서 붙인 이름을 __ (밑줄 두 개) 로 이어서 지목함.
#   힌트: "prep__num" 처럼.
param_grid = {
    "____": scalers,
    "____": encoders,
}


gs = GridSearchCV(pipe, param_grid, cv=5).fit(train_df[MEAS + ["island"]], y_train)

print(f"best_score_  : {gs.best_score_:.4f}")
best_names = {k: type(v).__name__ for k, v in gs.best_params_.items()}
print("best_params_ :", best_names)

pd.DataFrame(gs.cv_results_)[["param_prep__num", "param_prep__cat",
                              "mean_test_score", "rank_test_score"]]

### 답할 것

| | |
|---|---|
| 1위(rank 1) 조합 개수 | |
| `best_score_` | |
| `best_params_` | |

**예측 회수** — 1위가 몇 개일 거라고 적었나? 맞았나?

- 답:

**Q3E-1.** GridSearchCV가 `best_params_`로 **하나**를 찍어줬음.
그런데 표를 보면 1위가 여러 개임.

> **찍힌 하나가 나머지보다 더 좋다는 뜻인가?**

- 답:

**Q3E-2.** 표를 **세로로** 볼 것. 같은 스케일러 안에서 인코더만 바꾸면 숫자가 어떻게 되나?

- 답:

**Q3E-3.** Q3에서는 한 번만 비교했음. 지금은 세 스케일러 전부에서 확인했음.
그럼 이제 **"One-Hot과 Ordinal은 항상 같다"**고 말해도 되나?

- 답:

**Q3E-4.** `best_score_`가 나왔음. 그럼 **우리 모델의 최종 성능은 이 숫자**라고 말해도 되나?

- 답:

**Q3E-5.** 마지막으로 하나만 더.

> GridSearchCV는 **무엇을 기준으로** best라고 했을까?

우리가 `scoring=`을 지정한 적이 있나? 없다면 뭘 기준으로 순위를 매긴 걸까?

- 답:

---
## Q4. 순서를 맞춰라  `[키보드 금지]`

코드 조각 다섯 개가 섞여 있음. **하나는 쓰면 안 되는 함정 카드임.**

```
(가)  scaler.fit(X_train)
(나)  X_test_scaled  = scaler.transform(X_test)
(다)  model.fit(X_train_scaled, y_train)
(라)  X_train_scaled = scaler.transform(X_train)
(마)  scaler.fit(X_test)
```

### 답할 것 (코드 없이)

**Q4-1.** 빼야 할 함정 카드는? 그리고 **왜** 빼야 하나?

- 답:

**Q4-2.** 나머지 네 개를 올바른 순서로 배열할 것.
**순서가 반드시 정해지는 쌍**과 **바꿔도 되는 쌍**을 구분해서 적을 것.

- 답: ( ) → ( ) → ( ) → ( )
- 반드시 앞에 와야 하는 것:
- 바꿔도 되는 것:

**Q4-3.** (가)와 (라)를 합쳐서 `scaler.fit_transform(X_train)`으로 써도 되나?
그럼 (나)도 `fit_transform`으로 바꿔도 되나?

- 답:

**Q4-4.** 이번엔 함정 카드 말고, **깜빡 잊는 경우**를 생각해볼 것.

실수로 **(나)를 통째로 빼먹고** 실행했다고 하자.
즉 `X_train`은 스케일링해서 학습했는데, `X_test`는 **원본 그대로** 예측에 넣은 것.

- 에러가 날까, 그냥 돌아갈까?
- 돌아간다면 결과는 어떻게 될 것 같나?

> 힌트: 모델이 학습한 숫자는 대략 `-2 ~ 2` 범위였음.
> 그런데 원본 `body_mass_g`는 몇천 단위임. 모델은 그 숫자를 뭐라고 읽을까?

- 답:

> 이 예측은 **Q5에서 확인함.** 지금 적어둘 것.

---
## Q5. 가장 위험한 버그

같은 데이터로 네 사람이 코드를 짰음. 전부 **에러 없이 돌아감.**

이번엔 봉인해둔 test를 열지 않음. `train` 안에서 다시 나눠서 비교함.

```
train 273  =  diagnostic_train 218  +  diagnostic_val 55
```

**Q4 에서 종이로 배열한 그 순서**를 이번엔 직접 코드로 씀. 그게 `내 코드`임.
그리고 다른 세 사람(X, Y, Z)이 짠 것과 나란히 놓고 봄.

네 사람의 `diagnostic_val` 점수는 이렇게 나옴.

```
1.0000   1.0000   0.3636   1.0000
```

**[예측]** 아래 셀을 채우기 전에 먼저 적을 것.

| | 내 답 |
|---|---|
| `내 코드`는 몇 점이 나올 것 같음? | |
| `0.3636` 은 누구일 것 같음? | |
| 나머지 셋 중 **틀린 게 있을 것 같음?** | |

> 셋 다 적었으면 아래를 채우고 실행할 것.

In [ ]:
# TODO: Q4 에서 배열한 순서 그대로 코드를 쓸 것.
#   scaler 를 diagnostic_train 에 fit
#   -> diagnostic_train 과 diagnostic_val 둘 다 transform
#   -> 모델 학습 -> diagnostic_val 예측
#   예측 결과는 pred_mine 에 담을 것.
#
#   여기서는 Pipeline 을 쓰지 않고 손으로 씁니다. 손으로 하면 뭐가 위험한지 보려고요.

scaler = ...
fitted = ...
pred_mine = ...


# ↓ 아래는 다른 세 사람이 짠 것. 직접 짜지 말고, 위 코드와 비교하면서 읽을 것.

pred_X = fitted.predict(StandardScaler().fit_transform(diag_val_df[MEAS]))
pred_Y = fitted.predict(diag_val_df[MEAS].values)

scaler_Z = StandardScaler().fit(train_df[MEAS])
fitted_Z = KNeighborsClassifier(5).fit(scaler_Z.transform(diag_train_df[MEAS]), y_diag_train)
pred_Z   = fitted_Z.predict(scaler_Z.transform(diag_val_df[MEAS]))


for name, pred in [("내 코드", pred_mine), ("X", pred_X), ("Y", pred_Y), ("Z", pred_Z)]:
    print(f"  {name:6s} {accuracy_score(y_diag_val, pred):.4f}")

### 답할 것

**예측 회수**

| | 내 예측 | 실제 |
|---|---|---|
| `내 코드` 점수 | | |
| `0.3636` 은 누구 | | |
| 나머지 셋 중 틀린 게 있나 | | |

**Q5-0.** `내 코드`, X, Y, Z 중 **올바른 절차는 몇 개**였나?

- 답:

**Q5-1.** `0.3636` 이 나온 사람은 어디가 틀렸나?

- 답:

**Q5-2.** `내 코드`와 **점수가 같은데도** 틀린 사람이 있음. 누구고, 어디가 틀렸나?

- 답:

**Q5-3.** 또 한 명 있음. 그 사람은 **위 사람과 다른 종류의 실수**인가?

- 답:

**Q5-4.** 셋이 `1.0000`으로 같은데 그중 둘이 틀렸음. **무엇을 봐야** 구분할 수 있었나?

> **점수만 보고 골라낼 수 있었나?** 이게 오늘 실습의 핵심 질문임.

- 답:

---
## 오늘 정리

### 숫자 채우기

| 방법 | CV 평균 | diagnostic_val |
|---|---|---|
| 그대로 (스케일 안 맞춤) | | - |
| 스케일 맞춤 | | - |
| island 숫자 하나 | | - |
| island One-Hot | | - |
| Q5 A / B / C / D | - | / / / |

### 한 문장으로

1. Q1 (관찰):
2. Q2 (스케일):
3. Q3 (표현):
4. Q4 (절차):
5. Q5 (보이지 않는 오류):

### 오늘 부딪힌 벽

1-2에서 했던 것처럼, **답을 못 한 질문**을 적을 것.

| # | 벽 | 어디서 나왔나 |
|---|---|---|
| 1 | | Q3 |
| 2 | | Q5 |
| 3 | | |

> 힌트: 이번에도 **"점수가 낮아서"** 막힌 게 아님.

---

### 그런데 하나 남음

오늘 모델 원리는 몰라도 됐음. 그런데 하나는 알게 됐음.

> **모델에 무엇을 어떻게 넣었는지가 결과를 바꿈.**

그리고 오늘 비교는 전부 **Accuracy** 하나로 했음.

> **근데 이 숫자 하나만으로 모델을 평가해도 충분할까?**

그 질문은 **3-2회차**에서 이어감.